# TP2 — 01 : Ingestion dans MongoDB et simulation des paiements

**Binôme : Enora et Edwin** · Projet BigData Commerce Analytics

Nous récupérons les huit ressources DummyJSON, importons les lignes de commandes du jour 1 et créons un paiement simulé pour chaque panier. L'API et les données sont fictives. Les résultats observés dépendent de la date de collecte.

Avant l'exécution, démarrer MongoDB, ouvrir Jupyter depuis le dossier `jour-2` et choisir le noyau **Python (TP2 Big Data)**. Les écritures utilisent des identifiants et des mises à jour `upsert` : relancer le notebook avec les mêmes données ne crée pas de doublons.

## 1. Connexion à MongoDB

    La base `commerce` contient les collections brutes. Le résultat `ok: 1.0` confirme que MongoDB répond.

In [1]:
from pathlib import Path
import random
import pandas as pd
import requests
from pymongo import MongoClient

client = MongoClient("mongodb://127.0.0.1:27017", serverSelectionTimeoutMS=5000)
print("Connexion MongoDB :", client.admin.command("ping"))
db = client["commerce"]
print("Base sélectionnée :", db.name)

Connexion MongoDB : {'ok': 1.0}
Base sélectionnée : commerce


## 2. Récupérer les huit ressources de l'API

    Le paramètre `limit=0` demande tous les éléments disponibles. Pour chaque ressource, nous vérifions le total annoncé puis insérons ou remplaçons chaque document selon son `id`. Un index unique protège cet identifiant dans chaque collection.

In [2]:
ressources = ["products", "carts", "users", "posts", "comments", "recipes", "quotes", "todos"]
suivi_api = []

for ressource in ressources:
    reponse = requests.get(
        "https://dummyjson.com/" + ressource,
        params={"limit": 0},
        timeout=60,
    )
    reponse.raise_for_status()
    resultat = reponse.json()
    documents = resultat[ressource]
    total_api = resultat["total"]
    assert len(documents) == total_api, "La réponse API est incomplète"

    collection = db["raw_" + ressource]
    collection.create_index("id", unique=True)
    for document in documents:
        collection.replace_one({"id": document["id"]}, document, upsert=True)

    total_mongodb = collection.count_documents({})
    assert total_mongodb == total_api, "La collection ne correspond pas au total actuel de l'API"
    suivi_api.append({
        "ressource": ressource,
        "collection": collection.name,
        "total_api": total_api,
        "total_mongodb": total_mongodb,
    })

display(pd.DataFrame(suivi_api))

,ressource,collection,total_api,total_mongodb
0,products,raw_products,194,194
1,carts,raw_carts,208,208
2,users,raw_users,208,208
3,posts,raw_posts,251,251
4,comments,raw_comments,340,340
5,recipes,raw_recipes,50,50
6,quotes,raw_quotes,1454,1454
7,todos,raw_todos,254,254


## 3. Importer les commandes du jour 1

    Une commande comporte plusieurs lignes de produits. Nous ajoutons `numero_ligne` pour identifier chaque ligne du fichier, car un même produit peut apparaître plusieurs fois dans un panier.

In [3]:
chemin_csv = Path("../jour-1/data/processed/commandes_ecommerce.csv")
commandes = pd.read_csv(chemin_csv)
commandes.insert(0, "numero_ligne", range(1, len(commandes) + 1))

print("Nombre de lignes :", len(commandes))
print("Nombre de paniers distincts :", commandes["cart_id"].nunique())
print("Colonnes du fichier source :", len(commandes.columns) - 1)
print("Marques manquantes :", commandes["brand"].isna().sum())
print("Lignes répétant un couple panier / produit :", commandes.duplicated(["cart_id", "product_id"]).sum())
display(commandes[["numero_ligne", "cart_id", "user_id", "product_id", "quantity", "discounted_line_total", "order_date"]].head())

Nombre de lignes : 800
Nombre de paniers distincts : 208
Colonnes du fichier source : 16
Marques manquantes : 377
Lignes répétant un couple panier / produit : 12


,numero_ligne,cart_id,user_id,product_id,quantity,discounted_line_total,order_date
0,1,1,1,162,4,105.41,2025-03-02
1,2,1,1,113,3,10547.97,2025-06-13
2,3,1,1,122,3,839.76,2025-09-15
3,4,1,1,138,2,17.67,2025-09-20
4,5,2,2,86,5,93.12,2025-11-28


In [4]:
assert len(commandes) == 800
assert commandes["cart_id"].nunique() == 208
assert commandes["numero_ligne"].is_unique

# Les valeurs manquantes sont enregistrées comme null dans MongoDB.
documents_commandes = commandes.astype(object).where(commandes.notna(), None).to_dict("records")
db.raw_commandes.create_index("numero_ligne", unique=True)
for document in documents_commandes:
    db.raw_commandes.replace_one({"numero_ligne": document["numero_ligne"]}, document, upsert=True)

nombre_mongodb = db.raw_commandes.count_documents({})
print("Lignes dans commerce.raw_commandes :", nombre_mongodb)
assert nombre_mongodb == len(commandes)

Lignes dans commerce.raw_commandes : 800


### Contrôles de cohérence par panier

    Le client, la commande et le total doivent être identiques sur toutes les lignes d'un panier. Nous comparons aussi le total du panier avec la somme de ses lignes remisées.

    **Limite des données :** `order_date` est une date synthétique du jour 1. Elle varie selon les lignes de nombreux paniers. Pour obtenir une seule date de paiement par panier, nous retiendrons sa date la plus ancienne. L'évolution mensuelle sera donc une simulation, pas un historique réel.

In [5]:
par_panier = commandes.groupby("cart_id")
for colonne in ["order_id", "user_id", "cart_discounted_total"]:
    paniers_incoherents = int((par_panier[colonne].nunique(dropna=False) > 1).sum())
    print("Paniers avec plusieurs valeurs pour", colonne, ":", paniers_incoherents)
    assert paniers_incoherents == 0

dates_multiples = int((par_panier["order_date"].nunique() > 1).sum())
print("Paniers avec plusieurs dates synthétiques :", dates_multiples)

verification_montants = par_panier.agg(
    somme_lignes=("discounted_line_total", "sum"),
    montant_panier=("cart_discounted_total", "first"),
)
ecarts = (verification_montants["somme_lignes"] - verification_montants["montant_panier"]).abs()
print("Écart maximal entre lignes et montant du panier :", round(ecarts.max(), 2))
assert (ecarts < 0.02).all()

Paniers avec plusieurs valeurs pour order_id : 0
Paniers avec plusieurs valeurs pour user_id : 0
Paniers avec plusieurs valeurs pour cart_discounted_total : 0
Paniers avec plusieurs dates synthétiques : 207
Écart maximal entre lignes et montant du panier : 0.0


## 4. Simuler un paiement par panier

    Nous regroupons les lignes par `cart_id`, puis nous tirons un statut : **PAID** avec 70 % de probabilité, **REFUSED** avec 20 %, **PENDING** avec 10 %. La graine 42 rend la simulation reproductible. Ce sont des probabilités : les proportions observées ne sont pas obligatoirement exactement 70/20/10.

    Chaque paiement conserve le panier, la commande, le client, le montant remisé et la date synthétique retenue.

In [6]:
paniers = commandes.groupby("cart_id", as_index=False).agg(
    order_id=("order_id", "first"),
    user_id=("user_id", "first"),
    montant=("cart_discounted_total", "first"),
    date_paiement=("order_date", "min"),
)

generateur = random.Random(42)
statuts = ["PAID", "REFUSED", "PENDING"]
probabilites = [70, 20, 10]
db.raw_payments.create_index("cart_id", unique=True)

for paiement in paniers.to_dict("records"):
    paiement["payment_id"] = "PAY-" + str(paiement["cart_id"]).zfill(5)
    paiement["statut"] = generateur.choices(statuts, weights=probabilites, k=1)[0]
    db.raw_payments.replace_one({"cart_id": paiement["cart_id"]}, paiement, upsert=True)

nombre_paiements = db.raw_payments.count_documents({})
nombre_paniers_uniques = len(db.raw_payments.distinct("cart_id"))
print("Paniers :", len(paniers))
print("Paiements insérés :", nombre_paiements)
print("Paniers distincts dans les paiements :", nombre_paniers_uniques)
assert nombre_paiements == nombre_paniers_uniques == len(paniers) == 208

Paniers : 208
Paiements insérés : 208
Paniers distincts dans les paiements : 208


## 5. Analyser les paiements

    Seuls les paiements `PAID` sont encaissés. Les montants `REFUSED` et `PENDING` sont affichés pour comprendre la simulation, mais ne sont pas ajoutés au chiffre d'affaires encaissé.

In [7]:
paiements = pd.DataFrame(list(db.raw_payments.find({}, {"_id": 0})))
repartition = paiements.groupby("statut", as_index=False).agg(
    nombre=("cart_id", "size"),
    montant_total=("montant", "sum"),
)
repartition["proportion_pourcent"] = (repartition["nombre"] / len(paiements) * 100).round(2)
repartition["montant_total"] = repartition["montant_total"].round(2)
display(repartition)

montant_encaisse = paiements.loc[paiements["statut"] == "PAID", "montant"].sum()
print("Montant réellement encaissé :", round(montant_encaisse, 2))
print("Doublons de panier dans les paiements :", paiements["cart_id"].duplicated().sum())
assert paiements["cart_id"].duplicated().sum() == 0
assert set(paiements["cart_id"]) == set(commandes["cart_id"])

,statut,nombre,montant_total,proportion_pourcent
0,PAID,150,2669235.61,72.12
1,PENDING,21,363918.22,10.10
2,REFUSED,37,423555.75,17.79


Montant réellement encaissé : 2669235.61
Doublons de panier dans les paiements : 0


### Bilan

    Le contrôle du nombre de documents, l'index unique sur `cart_id` et la comparaison des identifiants vérifient qu'il existe une seule tentative par panier. Le montant encaissé correspond uniquement aux paiements acceptés. Les dates et les statuts sont synthétiques : cette analyse illustre un scénario de test.